In [1]:
import arcgis
import time
from arcgis.gis import GIS
from arcgis.gis import Item
from arcgis.apps.storymap import StoryMap

from typing import Set  # Import Set from typing
import re, json, csv

import pandas as pd
import os
import logging
import requests

# Set Pandas dataframe display options
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns',1000)

In [3]:
agoNotebook = False
# Print the version of the arcgis module
print(f"Running ArcGIS API for Python version: {arcgis.__version__}")

# Define the GIS
if agoNotebook == False:
    import keyring
    service_name = "system" # Use the default local credential store
    success = False # Set initial state

    # Ask for the username
    while success == False:
        username_for_keyring = input("Enter your ArcGIS Online username:") # If you are using VS Code, the text input dialog box appears at the top of the window
        # Get the credential object
        credential = keyring.get_credential(service_name, username_for_keyring)
        # Check if the username is in the credential store
        if credential is None:
            print(f"'{username_for_keyring}' is not in the local system's credential store. Try another username.")
        # Retrieve the password, login and set the GIS portal
        else:
            password_from_keyring = keyring.get_password("system", username_for_keyring)
            portal_url = 'https://www.arcgis.com'  
            gis = GIS(portal_url, username=username_for_keyring, password=password_from_keyring)
            success = True
            # Print a success message with username and user's organization role
            print("Successfully logged in as: " + gis.properties.user.username, "(role: " + gis.properties.user.role + ")")
else:
    gis = GIS("home")

Running ArcGIS API for Python version: 2.4.2
Successfully logged in as: dasbury_storymaps (role: org_admin)


In [4]:
classic_maptour_id = "20fd39888a444629bc8e40d9b6ac38cc"
classic_maptour_webmap = ""
classic_maptour_featureCollection = ""
classic_maptour_featureSet = ""

In [5]:
# Retrieve the JSON data for the classic MapTour item
classic_item = gis.content.get(classic_maptour_id)
classic_item_json = classic_item.get_data()
# import pprint
# pprint.pprint(classic_item_json)  # Optional: inspect structure

# Find the webmap ID referenced in the item JSON (usually in 'values' > 'webmap')
webmap_id = None
if 'values' in classic_item_json and 'webmap' in classic_item_json['values']:
    webmap_id = classic_item_json['values']['webmap']
    print(f"Found webmap ID: {webmap_id}")
else:
    print("Webmap ID not found in item JSON.")

# Download the webmap's JSON data
classic_maptour_webmap = None
if webmap_id:
    webmap_item = gis.content.get(webmap_id)
    classic_maptour_webmap = webmap_item.get_data()
    # pprint.pprint(classic_maptour_webmap)  # Optional: inspect structure
else:
    print("Cannot retrieve webmap JSON without webmap ID.")

# Parse the webmap JSON to get the featureCollection and featureSet
classic_maptour_featureCollection = None
classic_maptour_featureSet = None
if classic_maptour_webmap:
    # Look for operationalLayers with type 'Feature Layer' or 'featureCollection'
    layers = classic_maptour_webmap.get('operationalLayers', [])
    for layer in layers:
        # Check for featureCollection
        if 'featureCollection' in layer:
            classic_maptour_featureCollection = layer['featureCollection']
            print("Found featureCollection in webmap.")
            # Check for featureSet inside featureCollection
            if 'layers' in classic_maptour_featureCollection:
                for fc_layer in classic_maptour_featureCollection['layers']:
                    if 'featureSet' in fc_layer:
                        classic_maptour_featureSet = fc_layer['featureSet']
                        print("Found featureSet in featureCollection.")
                        break
            break
    if not classic_maptour_featureCollection:
        print("No featureCollection found in webmap.")
    if not classic_maptour_featureSet:
        print("No featureSet found in featureCollection.")
else:
    print("Webmap JSON not loaded.")

Found webmap ID: 680415c1370949f28bf03879cce97831
Found featureCollection in webmap.
Found featureSet in featureCollection.
Found featureCollection in webmap.
Found featureSet in featureCollection.


In [ ]:
import os
import requests

# Create directories if they don't exist
pics_dir = "mapTourTest/pics"
thumbs_dir = "mapTourTest/thumbs"
os.makedirs(pics_dir, exist_ok=True)
os.makedirs(thumbs_dir, exist_ok=True)

# Download images from each feature in classic_maptour_featureSet
if classic_maptour_featureSet and "features" in classic_maptour_featureSet:
    for i, feature in enumerate(classic_maptour_featureSet["features"]):
        # Download main image
        pic_url = feature["attributes"].get("pic_url")
        if pic_url:
            pic_filename = os.path.join(pics_dir, f"pic_{i}.jpg")
            try:
                response = requests.get(pic_url, timeout=10)
                if response.status_code == 200:
                    with open(pic_filename, "wb") as f:
                        f.write(response.content)
                    print(f"Downloaded: {pic_filename}")
                else:
                    print(f"Failed to download {pic_url}: {response.status_code}")
            except Exception as e:
                print(f"Error downloading {pic_url}: {e}")

        # Download thumbnail image
        thumb_url = feature["attributes"].get("thumb_url")
        if thumb_url:
            thumb_filename = os.path.join(thumbs_dir, f"thumb_{i}.jpg")
            try:
                response = requests.get(thumb_url, timeout=10)
                if response.status_code == 200:
                    with open(thumb_filename, "wb") as f:
                        f.write(response.content)
                    print(f"Downloaded: {thumb_filename}")
                else:
                    print(f"Failed to download {thumb_url}: {response.status_code}")
            except Exception as e:
                print(f"Error downloading {thumb_url}: {e}")
else:
    print("classic_maptour_featureSet['features'] not found or empty.")

In [20]:
print(gis.url)

https://Story.maps.arcgis.com


In [24]:
import uuid
import json
import requests
from arcgis.apps.storymap import StoryMap
username = gis.properties.user.username

# Step 1: Create a new StoryMap draft (using ArcGIS API for Python)
storymap = StoryMap(gis=gis)
storymap_item = storymap.save(publish=True)
storymap_id = storymap_item.itemid
print(f"Created StoryMap with ID: {storymap_id}")

# Step 2: Upload images to the StoryMap resources endpoint using REST API
pics_dir = "mapTourTest/pics"
pic_files = [f for f in os.listdir(pics_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
image_resource_map = {}  # Map pic filename to resourceId
portal_url = gis._portal.url if hasattr(gis, '_portal') else gis.url
# add_resource_url = f"{portal_url}/sharing/rest/content/items/{storymap_id}/addResource"
add_resource_url = f"https://www.arcgis.com/sharing/rest/content/users/{username}/items/{storymap_id}/addResources"
for file in pic_files:
    file_path = os.path.join(pics_dir, file)
    with open(file_path, "rb") as img_file:
        files = {"file": (file, img_file)}
        params = {
            "f": "json",
            "token": gis._con.token,
            "fileName": file
        }
        response = requests.post(add_resource_url, files=files, data=params)
        if response.status_code == 200 and response.json().get("success"):
            image_resource_map[file] = file  # Use filename as resourceId for mapping
            print(f"Uploaded resource: {file}")
        else:
            print(f"Failed to upload resource: {file}. Full response: {response.text}")

# Step 3: Build StoryMap JSON mimicking graves-tour.json schema
def build_tourmap_json(feature_set, image_resource_map):
    tour_map_node_id = "n-tourMap"
    tour_node_id = "n-tour"
    nodes = {}
    resources = {}
    geometries = {}
    places = []
    for i, feature in enumerate(feature_set["features"]):
        geom_id = str(uuid.uuid4())
        resource_id = str(uuid.uuid4())
        # Geometry
        geometries[geom_id] = {
            "id": geom_id,
            "type": "POINT_NUMBERED_TOUR",
            "nodes": [{"long": feature["geometry"]["x"], "lat": feature["geometry"]["y"]}]
        }
        # Image resource
        pic_url = feature["attributes"].get("pic_url")
        pic_filename = f"pic_{i}.jpg"
        resource_name = image_resource_map.get(pic_filename, pic_filename)
        resources[resource_id] = {
            "type": "image",
            "data": {
                "resourceId": resource_name,
                "provider": "item-resource",
                "height": 1024,
                "width": 1024
            }
        }
        # Place (tour node)
        places.append({
            "id": str(uuid.uuid4()),
            "featureId": geom_id,
            "contents": [feature["attributes"].get("description","")],
            "media": resource_id,
            "title": feature["attributes"].get("name","")
        })
    # Build tour-map node
    nodes[tour_map_node_id] = {
        "type": "tour-map",
        "data": {
            "geometries": geometries,
            "mode": "2d",
            "basemap": {"type": "name", "value": "worldImagery"}
        }
    }
    # Build tour node
    nodes[tour_node_id] = {
        "type": "tour",
        "data": {
            "type": "guided-tour",
            "subtype": "media-focused",
            "narrativePanelPosition": "start",
            "map": tour_map_node_id,
            "places": places,
            "narrativePanelSize": "medium",
            "accentColor": "#f9f794"
        }
    }
    # Build root node
    root_id = str(uuid.uuid4())
    nodes[root_id] = {
        "type": "story",
        "data": {"storyTheme": str(uuid.uuid4())},
        "children": [tour_node_id]
    }
    return {
        "root": root_id,
        "nodes": nodes,
        "resources": resources
    }

# Step 4: Create the StoryMap JSON and save to file
new_storymap_json = build_tourmap_json(classic_maptour_featureSet, image_resource_map)
with open("graves-tour-generated.json", "w") as f:
    json.dump(new_storymap_json, f, indent=2)
print("StoryMap JSON generated and saved as graves-tour-generated.json")

Created StoryMap with ID: 5e13a17fa7954de99ce513f7c9e66cc6
Uploaded resource: pic_0.jpg
Uploaded resource: pic_1.jpg
Uploaded resource: pic_10.jpg
Uploaded resource: pic_11.jpg
Uploaded resource: pic_12.jpg
Uploaded resource: pic_13.jpg
Uploaded resource: pic_14.jpg
Uploaded resource: pic_15.jpg
Uploaded resource: pic_16.jpg
Uploaded resource: pic_17.jpg
Uploaded resource: pic_18.jpg
Uploaded resource: pic_19.jpg
Uploaded resource: pic_2.jpg
Uploaded resource: pic_20.jpg
Uploaded resource: pic_21.jpg
Uploaded resource: pic_22.jpg
Uploaded resource: pic_3.jpg
Uploaded resource: pic_4.jpg
Uploaded resource: pic_5.jpg
Uploaded resource: pic_6.jpg
Uploaded resource: pic_7.jpg
Uploaded resource: pic_8.jpg
Uploaded resource: pic_9.jpg
StoryMap JSON generated and saved as graves-tour-generated.json
